# Reinforcement learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/intro/11_reinforcement_learning.ipynb)

Official API intro to reinforcement learning as a *planner*: a `StochasticPlanningProblem` states the task (plant, cost, box, exit rule, and what is random), `MonteCarloEvaluator` scores any controller on it, and `ReinforcementLearningPlanner` learns a neural feedback law $u = \pi_\theta(x)$ that comes back as an ordinary controller block. Because the learned law is a `System`, the closed loop compiles, linearizes and differentiates like everything else in minilink; the last section shows that off.

**Scripts for depth:** `examples/demos/rl/` (pendulum, cart-pole, car on a circuit, rocket landing, drone)

**Heavier labs:** [`teaching/drone_ppo_learn_to_fly.ipynb`](../teaching/drone_ppo_learn_to_fly.ipynb) (the Gymnasium + stable-baselines3 path) · [`experimental/rl/RL_README.md`](../../experimental/rl/RL_README.md) (tuning lessons)

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## The plant and the cost

A torque-limited pendulum ($\theta = 0$ hanging, $\theta = \pi$ upright) with less torque than gravity, so reaching the top takes a pumping motion. The cost is the textbook pair $J = \int g\,dt + h(x_f)$; the running cost is periodic in the angle, zero upright and two hanging. It is written with `jax.numpy` so it traces inside the compiled rollouts.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from minilink import CostFunction, DiagramSystem, Pendulum, PendulumWithNoisePort
from minilink.analysis import bode, plot_pzmap
from minilink.control import NeuralPolicyController, StateFeedbackController, angle_features
from minilink.planning import (
    MonteCarloEvaluator,
    ReinforcementLearningPlanner,
    StochasticPlanningProblem,
    Uniform,
)

TORQUE = 4.0  # Nm, below m g l = 9.81 Nm
DT = 0.05  # control period of the learned law


def torque_limited_pendulum(cls=Pendulum):
    plant = cls()
    plant.inputs["u"].lower_bound = np.array([-TORQUE])
    plant.inputs["u"].upper_bound = np.array([TORQUE])
    plant.state.lower_bound = np.array([-4 * np.pi, -20.0])  # the training box
    plant.state.upper_bound = np.array([4 * np.pi, 20.0])
    return plant


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        theta, dtheta = x
        return (1.0 + jnp.cos(theta)) + 0.01 * dtheta**2 + 0.01 * u[0] ** 2

    def h(self, x, t=0.0, params=None):
        return 0.0


plant = torque_limited_pendulum()
cost = SwingUpCost()
X_UP = np.array([np.pi, 0.0])

## A stochastic planning problem

`PlanningProblem` is deterministic: one start, one plant. `StochasticPlanningProblem` keeps the same spine and adds what is random — here the initial state, drawn uniformly over the whole circle with small rates — and the criterion (the expected cost). Two more declarations settle what episodes mean:

- `tf = inf` makes it an infinite-horizon task; the planners then pick a discount (or read `cost.discount_rate`) and an episode length.
- The state bounds are the allowed box $X$. Leaving it is a constraint violation whose price is declared **on the problem** (`on_exit`, `exit_cost`), never hidden in the cost function, so trajectory optimization, dynamic programming and reinforcement learning score the same trajectory the same way. Here we keep the default (`"infeasible"`, no price): an RL episode that leaves the box simply ends.

The same object answers `sample_x0` for Monte Carlo and `nominal()` for the deterministic planners.

In [ ]:
problem = StochasticPlanningProblem(
    plant,
    cost=cost,
    tf=np.inf,
    x0_distribution=Uniform([-np.pi, -1.0], [np.pi, 1.0]),
)
print("horizon:", problem.horizon_kind(), "| exit rule:", problem.on_exit, "| x_start =", problem.x_start)
print("three draws of x0:\n", np.round(problem.sample_x0(0, n=3), 2))
print("nominal problem:", type(problem.nominal()).__name__, "from", problem.nominal().x_start)

## Monte Carlo evaluation of any controller

`MonteCarloEvaluator` closes the loop with *any* state-feedback block, draws starts from the problem, and reports the distribution of $J$: mean, spread, worst case, and the failure rate (trials that left the box). It is the same yardstick for a hand-tuned law and for a learned one.

As a baseline, a PD law about the upright equilibrium, $u = -K (x - x_{up})$, with the torque clipped to the actuator limit. It balances the pendulum when it starts near the top and can do nothing from below.

In [ ]:
pd_ctl = StateFeedbackController(K=[[30.0, 8.0]], xbar=X_UP)
evaluator = MonteCarloEvaluator(problem, dt=DT, n_trials=100, seed=1)

report_pd = evaluator.evaluate(pd_ctl)
print("PD law:", report_pd)
print("best 10 trials (started near the top):", np.round(np.sort(report_pd.J)[:10], 2))

## Reinforcement learning as a planner

`ReinforcementLearningPlanner` reads the problem the way `DynamicProgrammingPlanner` does. It compiles the plant with the JAX backend, simulates many plants in parallel inside one `lax.scan`, and updates a neural policy with the chosen algorithm (`"ppo"` on-policy, `"sac"` off-policy; each algorithm is one file, the rest is shared). The policy sees *features* of the state — here the angle as $(\cos\theta, \sin\theta)$ and a scaled rate, via `angle_features` — but the law is still $u = \pi(x)$.

In [ ]:
planner = ReinforcementLearningPlanner(
    problem,
    dt=DT,
    features=angle_features(angles=[0], scales={1: 0.1}),
    hidden=(32, 32),
    algorithm="ppo",
    n_envs=64,
    n_steps=32,
    batch_size=256,
    learning_rate=3e-3,
    gamma=0.97,
    verbose=0,
)
plan = planner.solve(timesteps=120_000)
print(plan.metadata.message, f"in {plan.metadata.solve_time_s:.1f} s")
planner.plot_learning_curve()
plt.show()

## The learned law is a controller block

`get_controller()` returns a `NeuralPolicyController`: features, a multilayer perceptron whose weights live in `params["mlp"]`, and a map onto the actuator bounds. It draws its law with `plot_control_law` like LQR or value iteration, scores on the same Monte Carlo yardstick, and closes the loop with `@`.

In [ ]:
rl_ctl = planner.get_controller()
rl_ctl.plot_control_law(x_axis=0, y_axis=1, u_axis=0)  # torque vs (theta, dtheta)

report_rl = evaluator.evaluate(rl_ctl)
print("PD law:", report_pd)
print("RL law:", report_rl)

In [ ]:
plant.x0 = np.array([0.05, 0.0])  # hanging, a tiny tip to break the symmetry
cl_sys = rl_ctl @ plant
cl_sys.name = "Pendulum with the learned law"
cl_sys.plot_diagram()
traj = cl_sys.compute_trajectory(tf=10.0, dt=0.01)
cl_sys.plot_trajectory(traj)
cl_sys.animate(traj)

## Everything is a System: compile, linearize, differentiate

The closed loop `rl_ctl @ plant` is a diagram like any other, so the analysis tools apply to the *learned* law with no adapter. Linearizing at the upright equilibrium differentiates through the network and the plant in one JAX trace; the eigenvalues of $A$ tell whether the neural law stabilizes the top.

In [ ]:
lin = cl_sys.linearize(X_UP)  # exact Jacobians by autodiff through pi(x) and f(x, u)
poles = np.linalg.eigvals(lin.A())
print("closed-loop A at the top:\n", np.round(lin.A(), 3))
print("closed-loop poles:", np.round(poles, 3), "-> stable" if np.all(poles.real < 0) else "-> unstable")

**Disturbance rejection of a neural controller.** To ask a frequency-domain question we need an input to ask it from: the catalog `PendulumWithNoisePort` has a disturbance torque port `w`. Wire the same learned block around it by hand, expose `w` as the diagram's boundary input, and the frequency tools see one more `System`. Below: the poles of the open-loop plant (a real unstable pole at the top) against those of the closed loop with the RL law, the disturbance-to-angle Bode plots of both (the magnitudes are close because the learned law is not stiff at 4 Nm, the phases differ by the missing unstable pole), and the closed-loop pole-zero map.

In [ ]:
plant_w = torque_limited_pendulum(PendulumWithNoisePort)

loop = DiagramSystem()
loop.name = "RL law around the pendulum with a disturbance torque"
loop.add_subsystem(rl_ctl, "ctl")
loop.add_subsystem(plant_w, "plant")
loop.connect("plant", "y", "ctl", "x")
loop.connect("ctl", "u", "plant", "u")
loop.add_input_port("w", dim=1)
loop.connect("input", "w", "plant", "w")
loop.connect_new_output_port("plant", "y", "y")
loop.plot_diagram()

print("open-loop poles at the top:  ", np.round(np.linalg.eigvals(plant_w.linearize(X_UP).A()), 2))
print("closed-loop poles at the top:", np.round(np.linalg.eigvals(loop.linearize(X_UP).A()), 2))

w, mag_open, phase_open = bode(plant_w, X_UP, of=("y", 0), wrt="w")
_, mag_closed, phase_closed = bode(loop, X_UP, of=("y", 0), wrt="w")
fig, (ax_mag, ax_phase) = plt.subplots(2, 1, sharex=True, figsize=(7, 5))
ax_mag.semilogx(w, mag_open, label="open loop: w -> theta (unstable)")
ax_mag.semilogx(w, mag_closed, label="closed loop with the RL law")
ax_mag.set_ylabel("|G| [dB]")
ax_mag.legend()
ax_phase.semilogx(w, phase_open)
ax_phase.semilogx(w, phase_closed)
ax_phase.set_ylabel("phase [deg]")
ax_phase.set_xlabel("frequency [rad/s]")
for ax in (ax_mag, ax_phase):
    ax.grid(True, which="both", alpha=0.3)
plt.show()

plot_pzmap(loop, X_UP, of=("y", 0), wrt="w")

**Gradients through the physics.** The compiled diagram exposes the parametric step `x_{k+1} = \text{rk4}(x_k, u_k, t_k, \Delta t; p)` with `p` holding the controller weights *and* the plant parameters. A closed-loop rollout written as a `lax.scan` is then a function of both, and `jax.grad` gives, in one call, the sensitivity of the cost to the pendulum's mass and the gradient with respect to every network weight. One gradient step on the weights, through the plant, lowers the cost of the swing-up: the law learned by sampling can be fine-tuned by backpropagating through the dynamics.

In [ ]:
evaluator_jax = cl_sys.compile(backend="jax")
u_none = jnp.zeros(0)  # the closed loop has no boundary input
N = int(10.0 / DT)


def rollout_cost(params):
    def step(carry, k):
        x, t = carry
        u = rl_ctl.action(x, params["ctl"])  # the applied torque, for the cost
        J_k = cost.g(x, u, t) * DT
        x_next = evaluator_jax.rk4_step_trace_p(x, u_none, t, DT, params)
        return (x_next, t + DT), J_k

    (_, _), J_k = jax.lax.scan(step, (jnp.array([0.05, 0.0]), 0.0), jnp.arange(N))
    return jnp.sum(J_k)


params = jax.tree_util.tree_map(jnp.asarray, {"ctl": rl_ctl.params, "sys": plant.params})
J0, grads = jax.jit(jax.value_and_grad(rollout_cost))(params)
print(f"J of the swing-up from hanging: {float(J0):.3f}")
print("dJ/dm (pendulum mass):", float(grads["sys"]["m"]), "| dJ/dl (length):", float(grads["sys"]["l"]))
n_weights = sum(int(np.prod(g.shape)) for g in jax.tree_util.tree_leaves(grads["ctl"]))
print(f"gradient with respect to all {n_weights} policy weights: norm {float(jnp.sqrt(sum(jnp.sum(g**2) for g in jax.tree_util.tree_leaves(grads['ctl']))) ):.3f}")

# One gradient step on the weights, through the plant
lr = 0.02
tuned = dict(params)
tuned["ctl"] = jax.tree_util.tree_map(lambda w, g: w - lr * g, params["ctl"], grads["ctl"])
J1 = float(jax.jit(rollout_cost)(tuned))
print(f"after one gradient step through the dynamics: J = {J1:.3f} ({100 * (J1 - float(J0)) / float(J0):+.1f}%)")

## What to read next

- `examples/demos/rl/`: the five official scripts (pendulum, cart-pole, car on the MPC circuit, rocket landing, drone), each a `StochasticPlanningProblem` solved by the planner and scored by Monte Carlo.
- `experimental/rl/RL_README.md`: what it took to make each of them learn — the state-box exit exploit, periodic features, action normalization, reward scale, discount versus the task's time scale.
- `docs/plans/rl-planner-vision.md`: the design — one problem description, two verbs (`solve`, `evaluate`), one planner hosting an algorithm family.